In [3]:

import pandas as pd
path="https://drive.google.com/uc?export=download&id=1UwpW8GELggsVNj1IHtAo5rm4js_6b6PZ"
data = pd.read_csv(path)

data.columns

Index(['sepal.length', 'sepal.width', 'petal.length', 'petal.width',
       'variety'],
      dtype='object')

In [17]:
data.head()
train = data[['sepal.length', 'sepal.width', 'petal.length', 'petal.width']]
test = data["variety"]

In [21]:
train.head()
test.head()
test.unique()

array(['Setosa', 'Versicolor', 'Virginica'], dtype=object)

In [26]:

from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(train, test, train_size=0.9)


In [37]:

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense


In [44]:
model = Sequential([
    Input(shape=(4,)),
    Dense(16, activation="relu"),
    Dense(16, activation="relu"),
    Dense(3, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

In [39]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
ytrain = le.fit_transform(ytrain)
ytest = le.fit_transform(ytest)


In [ ]:

model.fit( xtrain, ytrain, epochs=50, shuffle=True, validation_split=0.1 )


In [58]:
from sklearn.metrics import accuracy_score
import numpy as np

result = model.predict(xtest)
result = np.argmax(result, axis=1)
print(accuracy_score(ytest, result))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1.0


In [62]:

model.save("ok.keras")


In [81]:
from tensorflow.keras.models import load_model
import numpy as np
test = load_model("ok.keras")

def iris_pre(sepal_length, sepal_width, petal_length, petal_width):
    output = ['Setosa', 'Versicolor', 'Virginica']
    inp = np.array([[sepal_length, sepal_width, petal_length, petal_width]])
    result = test.predict(inp)
    return output[np.argmax(result, axis=1)[0]]


In [82]:
from flask import Flask

app = Flask(__name__)

@app.route("/predict")
def predict(sepal_length, sepal_width, petal_length, petal_width):
    prediction = iris_pre(sepal_length, sepal_width, petal_length, petal_width)
    return {"prediction": prediction}
